In [1]:
import pandas as pd
from contextlib import ExitStack
from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window

In [6]:
# INVENTORY_BANDS_CSV = Path("/home/main/repositories/RemoteSensing/Qarterly_mosaics_30m_VelkaFatra/inventory.csv")
DISTURBANCE_MAIN_AREA=Path("/home/main/repositories/RemoteSensing/MachineLearning/dataset/cropped_annual_disturbances_1985_2023_slovakia.tif")
F_MASK_MAIN_AREA=Path("/home/main/repositories/RemoteSensing/MachineLearning/dataset/cropped_forest_mask_slovakia.tif")

DISTURBANCE_TEST_AREA=Path("/home/main/repositories/RemoteSensing/MachineLearning/dataset/TestArea/cropped_annual_disturbances_1985_2023_slovakia.tif")
F_MASK_TEST_AREA=Path("/home/main/repositories/RemoteSensing/MachineLearning/dataset/TestArea/cropped_forest_mask_slovakia.tif")

In [7]:
# -----------------------------------------

DISTURBANCE_TIF = DISTURBANCE_MAIN_AREA
FOREST_MASK_TIF = F_MASK_MAIN_AREA

# -----------------------------------------

FIRST_DISTURBANCE_YEAR = 1985  # year represented by disturbance band 1
BANDS = ("B02", "B03", "B04", "B08")
QUARTER = 3
DN_SCALE = 1000.0 # devide to convert to reflectance form digital number
FOREST_VALUE = 1
DISTURBANCE_VALUE = 1 # No disturbance = noData
TEST_FRACTION = 0.20
RANDOM_SEED = 42

In [8]:
rand = np.random.default_rng(RANDOM_SEED)

target_years = [2017, 2018, 2019, 2020, 2021, 2022, 2023]

In [9]:
with rasterio.open(FOREST_MASK_TIF) as src:
    forest_mask = src.read(1)
    print(src.bounds)



BoundingBox(left=4938475.150879223, bottom=2882183.1670521195, right=5013475.150879223, top=2957183.1670521195)


In [10]:
num_of_points_in_f_mask = np.sum(forest_mask==1)
print(num_of_points_in_f_mask)

4443496


In [11]:
with rasterio.open(DISTURBANCE_TIF) as src:
    disturbanance = src.read()

print(disturbanance.shape)
total_pos = 0

for i in range(1,11):
    disturbance_in_year = disturbanance[-i,:,:]
    pos_count = np.sum(disturbance_in_year==1)
    neg_count = num_of_points_in_f_mask-pos_count

    print(2023-i+1 ,pos_count, neg_count)
    total_pos+=pos_count

disturbance_2014_2023 = disturbanance[-10:, :, :] #disturbance_2014_2023[0,:,:] ->2014, [1,:,:] ->2015

print(f"total pos: {total_pos}")

print(disturbance_2014_2023.shape)


(39, 2500, 2500)
2023 17687 4425809
2022 47967 4395529
2021 21490 4422006
2020 29658 4413838
2019 40282 4403214
2018 58610 4384886
2017 43713 4399783
2016 28529 4414967
2015 37262 4406234
2014 39703 4403793
total pos: 364901
(10, 2500, 2500)


In [12]:
pos_no_disturbance=np.array(disturbance_2014_2023.shape[1:])
print(pos_no_disturbance)

num_of_disturbances = np.sum(disturbance_2014_2023[3:,:,:], axis=0)
print(num_of_disturbances.min(), num_of_disturbances.max())

pos_no_disturbance = np.zeros(disturbance_2014_2023.shape[1:])
pos_no_disturbance[num_of_disturbances==0] = 1
pos_no_disturbance[forest_mask==0] = 0
print(np.sum(pos_no_disturbance))


[2500 2500]
0 3
4191669.0


In [13]:
pos_one_disturbance = np.zeros(disturbance_2014_2023.shape[1:])
pos_one_disturbance[num_of_disturbances==1] = 1
pos_one_disturbance[np.sum(disturbance_2014_2023[0:3,:,:], axis=0)!=0] = 0 # remove ceses close to disturbance at begining
pos_one_disturbance[forest_mask==0] = 0
print(np.sum(pos_one_disturbance))


#test ds
d_copy = disturbance_2014_2023[-2,:,:]
d_copy[num_of_disturbances!=1] = 0
dist2022 = np.sum(d_copy)
print(dist2022)

235169.0
43291


In [14]:
disturbance_further_than_3_y = np.zeros(disturbance_2014_2023.shape)
for i in range(disturbance_2014_2023.shape[1]):
    for ii in range(disturbance_2014_2023.shape[2]):
        t = 3
        while t<disturbance_2014_2023.shape[0]:
            if(disturbance_2014_2023[t,i,ii]==1) and (np.sum( disturbance_2014_2023[t-3:t+1,i,ii] )==1):
                    disturbance_further_than_3_y[t,i,ii] = 1
                    t+=2
            else:
                t+=1
#disturbance_further_than_2_y[forest_mask==0] = 0
print(np.sum(disturbance_further_than_3_y))



254227.0


In [ ]:
#807550
#347214.0
#548006

#1 0 0 0 1 0

In [15]:
neg_in_distance_5_from_ever_pos = np.zeros(pos_no_disturbance.shape)
for i in range(neg_in_distance_5_from_ever_pos.shape[0]):
    for ii in range(neg_in_distance_5_from_ever_pos.shape[1]):
        if pos_no_disturbance[i,ii] and (np.sum( disturbance_2014_2023[:, max(0,i-5): min(i+6,disturbance_2014_2023.shape[1]), max(0,ii-5):min(ii+6, disturbance_2014_2023.shape[2])] ) == 0):
            neg_in_distance_5_from_ever_pos[i,ii] = 1
neg_in_distance_5_from_ever_pos[forest_mask==0] = 0

print(np.sum(neg_in_distance_5_from_ever_pos))

2199978.0


In [16]:

with open("/home/main/repositories/RemoteSensing/MachineLearning/dataset/pos_disturbance_further_than_3_y.csv", "w") as f:

    f.write("Year;PosX;PosY\n")

    for t in range(disturbance_further_than_3_y.shape[0]):
        for x in range(disturbance_further_than_3_y.shape[1]):
            for y in range(disturbance_further_than_3_y.shape[2]):
                if (disturbance_further_than_3_y[t,x,y]==1):
                    f.write(f"{t+2014};{x};{y}\n")

with open("/home/main/repositories/RemoteSensing/MachineLearning/dataset/pos_disturbance_one.csv", "w") as f:

    # year of detection
    f.write("Year;PosX;PosY\n")

    for x in range(pos_one_disturbance.shape[0]):
        for y in range(pos_one_disturbance.shape[1]):
            if (pos_one_disturbance[x,y]==1):
                year = np.argmax(disturbanance[:, x,y] == 1)
                f.write(f"{y+2014};{x};{y}\n")

with open("/home/main/repositories/RemoteSensing/MachineLearning/dataset/neg_no_disturbances.csv", "w") as f:
    f.write("PosX;PosY\n")
    for x in range(neg_in_distance_5_from_ever_pos.shape[0]):
        for y in range(neg_in_distance_5_from_ever_pos.shape[1]):
            if(neg_in_distance_5_from_ever_pos[x,y]==1):
                f.write(f"{x};{y}\n")
